In [45]:
from pyspark.sql import SparkSession
from pyspark import SparkConf
from pyspark.sql.functions import expr, col


conf = SparkConf()
conf.set("spark.driver.host", "spark-iceberg")
conf.set("spark.driver.bindAddress", "0.0.0.0")
conf.set("spark.driver.port", "4040")
conf.set("spark.blockManager.port", "4041")

# 0) Stop any existing Spark session
try:
    spark.stop()
except:
    pass

# 1) Build a fresh SparkSession with Iceberg/REST as spark_catalog
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
      .appName("IcebergBucketRefactor")
      .master("local[*]")
      # point default catalog at Iceberg REST
      .config("spark.sql.catalog.spark_catalog", "org.apache.iceberg.rest.RESTCatalog")
      .config("spark.sql.catalog.spark_catalog.uri", "http://rest:8181")
      .config("spark.sql.catalog.spark_catalog.default-namespace", "default")
      .getOrCreate()
)

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

# 2) Ensure the default namespace exists
spark.sql("CREATE NAMESPACE IF NOT EXISTS default")

# Load data
matches = spark.read.option("header","true") \
    .csv("/home/iceberg/data/matches.csv")
medals = spark.read.option("header","true") \
    .csv("/home/iceberg/data/medals.csv")
maps = spark.read.option("header","true") \
    .csv("/home/iceberg/data/maps.csv")
match_details = spark.read.option("header", "true") \
    .csv("/home/iceberg/data/match_details.csv")
medals_matches_players = spark.read.option("header","true") \
    .csv("/home/iceberg/data/medals_matches_players.csv")



25/07/05 23:08:23 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [17]:
# Task 1: broadcast join medals and matches

# Broadcast join
df = medals.join(maps, on="medal_id", how="left")
df.show()

[Stage 5:====================================>                      (5 + 3) / 8]

+----------+--------------------+-----------+----------+------------------+-------------------+------------+-------------+--------------+--------------------+----------+----------+--------------------+---------------+-----+
|  medal_id|          sprite_uri|sprite_left|sprite_top|sprite_sheet_width|sprite_sheet_height|sprite_width|sprite_height|classification|         description|      name|difficulty|            match_id|player_gamertag|count|
+----------+--------------------+-----------+----------+------------------+-------------------+------------+-------------+--------------+--------------------+----------+----------+--------------------+---------------+-----+
|2359847435|https://content.h...|        525|       375|                74|                 74|        1125|          899|      Breakout|Kill every member...|Extinction|        35|e6cda10a-0bbe-4e8...|       EcZachly|    1|
|2359847435|https://content.h...|        525|       375|                74|                 74|        1

In [25]:
# Task 

# 0) Constants
bucket_count=16

# 1) For each source CSV: read → temp view → CTAS bucketed table
for name, path in [
    ("match_details",        "/home/iceberg/data/match_details.csv"),
    ("matches",              "/home/iceberg/data/matches.csv"),
    ("medals_matches_players","/home/iceberg/data/medals_matches_players.csv"),
]:
    temp_view = f"{name}_raw"
    table_name = f"default.bucketed_{name}"
    
    # 3a) Read CSV with header into a temp view
    spark.read.option("header", "true") \
         .csv(path) \
         .createOrReplaceTempView(temp_view)
    
    # 3b) CTAS into an Iceberg table clustered (bucketed) on match_id
    spark.sql(f"""
      CREATE OR REPLACE TABLE {table_name}
      USING iceberg
      TBLPROPERTIES ('format-version'='2')
      CLUSTERED BY (match_id) INTO {bucket_count} BUCKETS
      AS
      SELECT * FROM {temp_view}
    """)

# 2) Read the three bucketed tables
md = spark.table("default.bucketed_match_details")
m  = spark.table("default.bucketed_matches")
mmp= spark.table("default.bucketed_medals_matches_players")

# 3) Join on match_id
joined_df = md.join(m,  "match_id") \
              .join(mmp,"match_id")


+--------------------+---------------+---------------------+------------+-----------------+--------+-----------------+------------------------+------------+---------------------------------+-----------------+----------------+-----------------------+-----------+--------------------------------+----------------+-------------------+---------------+-------------------+------------------+----------------------+--------------------------+-------------------------+------------------------+-------------------------+---------------------------+-------------------------------+--------------------------------+---------------------------+--------------------------------+-------------------------------+-------------------+--------------------+--------------------------+-------+-------+--------------------+------------+--------------------+--------------------+-------------+--------------------+--------------+---------+--------------+---------------+----------+-----+
|            match_id|player_gam

In [41]:
joined_df.columns

['match_id',
 'player_gamertag',
 'previous_spartan_rank',
 'spartan_rank',
 'previous_total_xp',
 'total_xp',
 'previous_csr_tier',
 'previous_csr_designation',
 'previous_csr',
 'previous_csr_percent_to_next_tier',
 'previous_csr_rank',
 'current_csr_tier',
 'current_csr_designation',
 'current_csr',
 'current_csr_percent_to_next_tier',
 'current_csr_rank',
 'player_rank_on_team',
 'player_finished',
 'player_average_life',
 'player_total_kills',
 'player_total_headshots',
 'player_total_weapon_damage',
 'player_total_shots_landed',
 'player_total_melee_kills',
 'player_total_melee_damage',
 'player_total_assassinations',
 'player_total_ground_pound_kills',
 'player_total_shoulder_bash_kills',
 'player_total_grenade_damage',
 'player_total_power_weapon_damage',
 'player_total_power_weapon_grabs',
 'player_total_deaths',
 'player_total_assists',
 'player_total_grenade_kills',
 'did_win',
 'team_id',
 'mapid',
 'is_team_game',
 'playlist_id',
 'game_variant_id',
 'is_match_over',
 'com

In [43]:
token='spree'

[col_ for col_ in joined_df.columns if f"{token}" in col_]


[]

In [51]:
from pyspark.sql.functions import broadcast, col

DATA_PATH='/home/iceberg/data'

# Load small lookup tables
medals = spark.read.csv(os.path.join(DATA_PATH, "medals.csv"), header=True)
maps = spark.read.csv(os.path.join(DATA_PATH, "maps.csv"), header=True)

# Load large tables (will be bucketed)
match_details = spark.table("default.bucketed_match_details")
matches = spark.table("default.bucketed_matches")
medals_matches_players = spark.table("default.bucketed_medals_matches_players")

# Step 1: Bucket join large tables
# First join match_details and matches
joined_md_m = match_details.join(matches, "match_id")

# Then join with medals_matches_players, but handle duplicate player_gamertag
joined_large = (joined_md_m
               .join(medals_matches_players.select("match_id", "medal_id", "count"), "match_id"))

# THESE LINES FULFILL THE REQUIREMENT:
joined_with_medals = joined_large.join(broadcast(medals), "medal_id", "left")
final_joined = joined_with_medals.join(broadcast(maps), "mapid", "left")

In [60]:
def question_1_most_kills_per_game(joined_df):
    """
    ✅ QUESTION 1: Which player averages the most kills per game?
    """
    from pyspark.sql.functions import avg, desc, col
    
    # THIS CODE ANSWERS THE QUESTION:
    # Use match_details.player_gamertag since that's where player_total_kills comes from
    result = (joined_df
              .groupBy("player_gamertag")  # Use original column name from bucketed_match_details
              .agg(avg("player_total_kills").alias("avg_kills"))
              .orderBy(desc("avg_kills"))
              .limit(5))
    
    return result

def question_2_most_played_playlist(joined_df):
    """
    ✅ QUESTION 2: Which playlist gets played the most?
    """
    from pyspark.sql.functions import count, desc, col
    
    # THIS CODE ANSWERS THE QUESTION:
    result = (joined_df
              .groupBy("playlist_id")  # Use original column name from bucketed_matches
              .agg(count("*").alias("play_count"))
              .orderBy(desc("play_count"))
              .limit(5))
    
    return result

def question_3_most_played_map(joined_df):
    """
    ✅ QUESTION 3: Which map gets played the most?
    """
    from pyspark.sql.functions import count, desc, col
    
    # THIS CODE ANSWERS THE QUESTION:
    result = (joined_df
              .groupBy("mapid", "map_name")  # Use aliased column name
              .agg(count("*").alias("map_play_count"))
              .orderBy(desc("map_play_count"))
              .limit(5))
    
    return result

def question_4_most_killing_spree_medals_by_map(joined_df):
    """
    ✅ QUESTION 4: Which map do players get the most Killing Spree medals on?
    """
    from pyspark.sql.functions import count, desc, col
    
    # THIS CODE ANSWERS THE QUESTION:
    result = (joined_df
              .filter(col("medal_name").like("%Killing Spree%"))  # Filter on medal name
              .groupBy("mapid", "map_name")  # Group by map info
              .agg(count("*").alias("killing_spree_count"))
              .orderBy(desc("killing_spree_count"))
              .limit(5))
    
    return result

In [61]:
question_1_most_kills_per_game(final_joined).show()

[Stage 62:=============================>                            (1 + 1) / 2]

+---------------+-----------------+
|player_gamertag|        avg_kills|
+---------------+-----------------+
|   gimpinator14|            109.0|
|  I Johann117 I|             96.0|
|BudgetLegendary|             83.0|
|      GsFurreal|             75.0|
|   TameablePoet|74.22429906542057|
+---------------+-----------------+



In [62]:
question_2_most_played_playlist(final_joined).show()

[Stage 71:==================================================>       (7 + 1) / 8]

+--------------------+----------+
|         playlist_id|play_count|
+--------------------+----------+
|f72e0ef0-7c4a-430...|   1565529|
|780cc101-005c-4fc...|   1116002|
|0bcf2be1-3168-4e4...|   1015496|
|c98949ae-60a8-43d...|    824932|
|2323b76a-db98-4e0...|    692342|
+--------------------+----------+



In [63]:
question_3_most_played_map(final_joined).show()

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `map_name` cannot be resolved. Did you mean one of the following? [`name`, `name`, `sprite_top`, `sprite_uri`, `sprite_left`].;
'Aggregate [mapid#5901, 'map_name], [mapid#5901, 'map_name, count(1) AS map_play_count#6846L]
+- Project [mapid#5901, medal_id#5922, match_id#5828, player_gamertag#5829, previous_spartan_rank#5830, spartan_rank#5831, previous_total_xp#5832, total_xp#5833, previous_csr_tier#5834, previous_csr_designation#5835, previous_csr#5836, previous_csr_percent_to_next_tier#5837, previous_csr_rank#5838, current_csr_tier#5839, current_csr_designation#5840, current_csr#5841, current_csr_percent_to_next_tier#5842, current_csr_rank#5843, player_rank_on_team#5844, player_finished#5845, player_average_life#5846, player_total_kills#5847, player_total_headshots#5848, player_total_weapon_damage#5849, ... 36 more fields]
   +- Join LeftOuter, (mapid#5901 = mapid#5822)
      :- Project [me

In [ ]:
question_4_most_killing_spree_medals_by_map(final_joined).show()

In [ ]:
requirement_5_sort_within_partitions(joined_df).show()

In [40]:
# Task 4: Answer questions

# Question 1: Which player averages the most kills per game?
question="Which player averages the most kills per game?"
print(question)
spark.sql("""
  SELECT 
      player_gamertag, 
      avg(player_total_kills) as avg_kils
  FROM default.bucketed_match_details
  group by player_gamertag
  order by 2 desc
  limit 1
""").show()

# Question: Which playlist gets played the most?
question="Which playlist gets played the most?"
print(question)
spark.sql("""
  SELECT 
      playlist_id, 
      count(1) as total_times_played
  FROM default.bucketed_match_details
  group by playlist_id
  order by 2 desc
  limit 1
""").show()

# Question: Which map gets played the most?
question="Which playlist gets played the most?"
print(question)
spark.sql("""
  SELECT 
      mapid, 
      count(1) as total_times_played
  FROM default.bucketed_match_details
  group by mapid
  order by 2 desc
  limit 1
""").show()

# Question: Which map do players get the most Killing Spree medals on?
question="Which map do players get the most Killing Spree medals on?"
print(question)
spark.sql("""
  SELECT
      playlist_id, 
      count(1) as total_times_played
  FROM default.bucketed_match_details
  group by playlist_id
  order by 2 desc
  limit 1
""").show()

Which player averages the most kills per game?
+---------------+--------+
|player_gamertag|avg_kils|
+---------------+--------+
|   gimpinator14|   109.0|
+---------------+--------+

Which playlist gets played the most?


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `playlist_id` cannot be resolved. Did you mean one of the following? [`match_id`, `team_id`, `did_win`, `current_csr`, `player_finished`].; line 3 pos 6;
'GlobalLimit 1
+- 'LocalLimit 1
   +- 'Sort [unresolvedordinal(2) DESC NULLS LAST], true
      +- 'Aggregate ['playlist_id], ['playlist_id, count(1) AS total_times_played#5141L]
         +- SubqueryAlias demo.default.bucketed_match_details
            +- RelationV2[match_id#5142, player_gamertag#5143, previous_spartan_rank#5144, spartan_rank#5145, previous_total_xp#5146, total_xp#5147, previous_csr_tier#5148, previous_csr_designation#5149, previous_csr#5150, previous_csr_percent_to_next_tier#5151, previous_csr_rank#5152, current_csr_tier#5153, current_csr_designation#5154, current_csr#5155, current_csr_percent_to_next_tier#5156, current_csr_rank#5157, player_rank_on_team#5158, player_finished#5159, player_average_life#5160, player_total_kil